<a href="https://colab.research.google.com/github/JoseAroudo/Compiladores/blob/main/TclTk_20_80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tcl/Tk esencial: el 20 % que explica el 80 %

Cuaderno práctico para aprender el núcleo de **Tcl** y usarlo después con **Tk**. Está diseñado para ejecutarse de arriba hacia abajo con un kernel Tcl.

## Objetivos

Al terminar podrá:

- leer Tcl como una secuencia de comandos y palabras;
- predecir las sustituciones `$`, `[]`, comillas y llaves;
- trabajar con expresiones, control, listas, diccionarios y procedimientos;
- crear una interfaz Tk con widgets, geometría y eventos;
- reconocer la arquitectura básica de programas como Picol y JimTcl.

## Preparación del kernel Tcl

En una terminal del Mac, dentro del mismo entorno de Python donde usa Jupyter:

```bash
python3 -m pip install tcl_kernel
python3 -m tcl_kernel.install
jupyter kernelspec list
jupyter lab
```

Abra este archivo y seleccione **Kernel → Change Kernel → Tcl** si Jupyter no lo selecciona automáticamente. El proyecto `tcl_kernel` indica que usa el Tcl incluido con Python mediante Tkinter.

> Nota: el kernel es suficiente para las secciones Tcl. Las ventanas Tk requieren Jupyter ejecutándose localmente en un entorno gráfico; normalmente aparecen como ventanas externas y no dentro de la celda.

In [ ]:
!apt-get update -qq
!apt-get install -y tcl

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tcl is already the newest version (8.6.14build1).
tcl set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 28 not upgraded.


In [ ]:
%%script tclsh

puts "Versión de Tcl: [info patchlevel]"
puts "Ejecutable: [info nameofexecutable]"
puts "Kernel listo: [expr {6 * 7}]"

Versión de Tcl: 8.6.14
Ejecutable: /usr/bin/tclsh
Kernel listo: 42


---
# 1. La idea que explica Tcl: todo es un comando

Una instrucción Tcl es una lista de palabras:

```tcl
comando argumento1 argumento2 ...
```

La primera palabra identifica el comando y las restantes son sus argumentos. El salto de línea o el punto y coma termina el comando. Incluso `if`, `while` y `proc` siguen este modelo.

In [ ]:
%%script tclsh

puts "Hola, Tcl"
set lenguaje Tcl
puts "Estoy aprendiendo $lenguaje"
puts [expr {10 + 20}]

Hola, Tcl
Estoy aprendiendo Tcl
30


## Experimento: un comando también devuelve un resultado

`set nombre valor` asigna y devuelve el valor. `set nombre` consulta la variable. El resultado de un comando se puede insertar en otro usando corchetes.

In [ ]:
set nombre "Ana"
puts "Valor devuelto por set: [set nombre]"
set longitud [string length $nombre]
puts "$nombre tiene $longitud caracteres"

# 2. El corazón de Tcl: sustitución y agrupación

Antes de invocar un comando, el intérprete separa sus palabras y realiza sustituciones:

| Forma | Efecto |
|---|---|
| `$variable` | sustituye el valor de una variable |
| `[comando]` | ejecuta el comando y sustituye su resultado |
| `"..."` | agrupa una palabra y permite sustituciones |
| `{...}` | agrupa una palabra y evita sustituciones inmediatas |
| `\` | escapa o da significado especial al carácter siguiente |

Las llaves no significan “bloque” por sí mismas: producen una sola palabra sin sustitución inmediata. El comando receptor decide qué hacer con ella.

In [ ]:
set persona Carlos
puts "1. Comillas: Hola $persona"
puts {2. Llaves: Hola $persona}
puts "3. Corchetes: 8 por 7 es [expr {8 * 7}]"
puts "4. Escape: precio \$25 y salto de línea\nsegunda línea"

### Prediga antes de ejecutar

¿Qué imprimirá cada línea? La habilidad central en Tcl consiste en distinguir **cuándo** se sustituye cada fragmento.

In [ ]:
set x 10
puts "A: $x"
puts {B: $x}
puts "C: [expr {$x + 5}]"
puts {D: [expr {$x + 5}]}

**Respuesta esperada:** `A: 10`, `B: $x`, `C: 15` y el texto literal `D: [expr {$x + 5}]`.

# 3. Expresiones y decisiones

Use `expr` para cálculos y coloque la expresión entre llaves. Es la forma segura, legible y eficiente:

```tcl
expr {$a + $b}
```

In [ ]:
set a 10
set b 3
puts "Suma: [expr {$a + $b}]"
puts "División entera: [expr {$a / $b}]"
puts "División real: [expr {$a / double($b)}]"
puts "¿a es mayor?: [expr {$a > $b}]"

In [ ]:
set nota 4.2
if {$nota >= 4.5} {
    puts "Excelente"
} elseif {$nota >= 3.0} {
    puts "Aprobó"
} else {
    puts "No aprobó"
}

Observe que `if` recibe palabras: una condición y uno o más guiones de código. El propio comando `if` evalúa esos argumentos en el momento apropiado. Por eso las llaves son esenciales.

# 4. Repetición: aprenda primero `foreach`

Tcl posee `for` y `while`, pero `foreach` es el ciclo que más aprovechará al procesar colecciones.

In [ ]:
set lenguajes {Tcl Python C Java}
foreach lenguaje $lenguajes {
    puts "Lenguaje: $lenguaje"
}

In [ ]:
puts "for:"
for {set i 1} {$i <= 5} {incr i} {
    puts "  $i al cuadrado = [expr {$i * $i}]"
}

puts "while:"
set n 3
while {$n > 0} {
    puts "  $n"
    incr n -1
}

# 5. Listas: la estructura de datos principal

Una lista es una secuencia de elementos. No la trate como texto separado manualmente: utilice los comandos `list`, `lappend`, `lindex`, `llength`, `lrange`, `lsearch` y `lsort`.

In [ ]:
set frutas [list manzana pera "mango maduro"]
lappend frutas naranja
puts "Lista: $frutas"
puts "Cantidad: [llength $frutas]"
puts "Primera: [lindex $frutas 0]"
puts "Última: [lindex $frutas end]"
puts "Sublista: [lrange $frutas 1 2]"
puts "Posición de pera: [lsearch -exact $frutas pera]"

In [ ]:
set numeros {8 2 10 3}
puts "Orden textual: [lsort $numeros]"
puts "Orden numérico: [lsort -integer $numeros]"

set total 0
foreach numero $numeros {
    incr total $numero
}
puts "Total: $total"

## Expansión de listas con `{*}`

`{*}` toma los elementos de una lista y los inserta como palabras independientes. Es la forma moderna de construir y ejecutar comandos sin recurrir a `eval`.

In [ ]:
set comando [list puts "Hola desde una lista-comando"]
puts "Representación: $comando"
{*}$comando

# 6. Diccionarios: registros con clave y valor

Use listas para secuencias y diccionarios para datos identificados por una clave.

In [ ]:
set estudiante [dict create \
    nombre "Laura" \
    edad 21 \
    carrera "Ingeniería de Sistemas"]

dict set estudiante semestre 5
puts "Nombre: [dict get $estudiante nombre]"
puts "Semestre: [dict get $estudiante semestre]"

dict for {clave valor} $estudiante {
    puts "$clave: $valor"
}

# 7. Procedimientos: dividir el programa en piezas

`proc nombre argumentos cuerpo` crea un nuevo comando Tcl. Sus variables son locales, salvo que se indique lo contrario.

In [ ]:
proc sumar {a b} {
    return [expr {$a + $b}]
}

proc saludar {nombre {mensaje "Bienvenido"}} {
    return "$mensaje, $nombre"
}

puts [sumar 10 20]
puts [saludar Ana]
puts [saludar Pedro "Buenos días"]

### Diseño recomendado

Prefiera procedimientos que reciban datos y devuelvan resultados. Reserve `global` para estado pequeño de una interfaz u otros casos conscientes.

In [ ]:
proc promedio {numeros} {
    if {[llength $numeros] == 0} {
        error "La lista no puede estar vacía"
    }

    set suma 0.0
    foreach n $numeros {
        set suma [expr {$suma + $n}]
    }
    return [expr {$suma / [llength $numeros]}]
}

puts "Promedio: [promedio {3.5 4.0 4.5}]"

# 8. Errores: `catch` y `error`

Un error no tiene que terminar el programa. `catch` ejecuta un guion protegido y permite inspeccionar el resultado.

In [ ]:
proc dividir {a b} {
    if {$b == 0} {
        error "No se puede dividir entre cero"
    }
    return [expr {$a / double($b)}]
}

if {[catch {dividir 10 0} resultado]} {
    puts "Error controlado: $resultado"
} else {
    puts "Resultado: $resultado"
}

---
# 9. Tk en tres ideas

Una interfaz Tk combina:

1. **Widgets:** `label`, `button`, `entry`, `text`, `frame`, etc.
2. **Geometría:** `pack`, `grid` o `place`.
3. **Eventos:** `-command` y `bind`.

La ventana principal se llama `.`. Un nombre como `.panel.guardar` describe el widget `guardar` dentro de `.panel`.

## Prueba opcional de Tk

Ejecute la celda siguiente solo en Jupyter local. `package require Tk` puede fallar en servidores sin pantalla. La ventana creada se cerrará al pulsar el botón.

In [ ]:
%%script tclsh

package require Tk
wm title . "Mi primera aplicación"
wm geometry . 400x200

label .titulo -text "Hola desde Tcl/Tk" -font {Helvetica 18 bold}
button .cerrar -text "Cerrar" -command {destroy .}

pack .titulo -pady 30
pack .cerrar

no display name and no $DISPLAY environment variable
invalid command name "wm"
invalid command name "wm"
invalid command name "label"
invalid command name "button"
invalid command name "pack"
invalid command name "pack"


## Formulario con `grid` y eventos

Si cerró la ventana anterior, reinicie el kernel antes de ejecutar esta celda. En aplicaciones reales, cada ejemplo se guardaría en un archivo `.tcl` y se ejecutaría con `wish archivo.tcl`.

In [ ]:
package require Tk
wm title . "Saludo"

set nombre ""

proc saludarGUI {} {
    global nombre
    if {[string trim $nombre] eq ""} {
        .resultado configure -text "Escriba un nombre"
    } else {
        .resultado configure -text "Hola, $nombre"
    }
}

label .nombreLabel -text "Nombre:"
entry .nombreEntry -textvariable nombre
button .saludar -text "Saludar" -command saludarGUI
label .resultado -text ""

grid .nombreLabel -row 0 -column 0 -padx 8 -pady 8 -sticky e
grid .nombreEntry -row 0 -column 1 -padx 8 -pady 8
grid .saludar -row 1 -column 0 -columnspan 2 -pady 8
grid .resultado -row 2 -column 0 -columnspan 2 -pady 8

bind .nombreEntry <Return> {saludarGUI}
bind . <Escape> {destroy .}
focus .nombreEntry

### `pack` frente a `grid`

- `pack`: excelente para franjas, paneles y widgets apilados.
- `grid`: excelente para formularios organizados en filas y columnas.
- No mezcle ambos administradores para hijos del **mismo contenedor**. Puede usar `pack` en la ventana y `grid` dentro de un `frame`.

# 10. Ejemplo integrador: contador Tcl/Tk

Este programa reúne estado, procedimientos, widgets, callbacks y contenedores. Ejecútelo tras reiniciar el kernel o copie el código en `contador.tcl` y use `wish contador.tcl`.

In [ ]:
%%writefile ejemplo.tcl
package require Tk
set contador 0

wm title . "Contador Tcl/Tk"
wm geometry . 320x210

proc actualizar {} {
    global contador
    .valor configure -text $contador
}

proc cambiar {cantidad} {
    global contador
    incr contador $cantidad
    actualizar
}

proc reiniciar {} {
    global contador
    set contador 0
    actualizar
}

label .titulo -text "Contador" -font {Helvetica 18 bold}
label .valor -text $contador -font {Helvetica 32}
frame .controles
button .controles.menos -text "−" -width 5 -command {cambiar -1}
button .controles.cero -text "Reiniciar" -command reiniciar
button .controles.mas -text "+" -width 5 -command {cambiar 1}

pack .titulo -pady {20 5}
pack .valor -pady 10
pack .controles.menos .controles.cero .controles.mas -side left -padx 5
pack .controles

bind . <Left> {cambiar -1}
bind . <Right> {cambiar 1}
bind . <Escape> {destroy .}

Writing ejemplo.tcl


---
# 11. Taller corto

Resuelva primero sin mirar las pistas:

1. Cree una lista con cinco notas y calcule promedio, menor y mayor.
2. Cree un diccionario para un curso con `nombre`, `grupo` y `estudiantes`.
3. Escriba `clasificarNota nota`, que devuelva `Excelente`, `Aprobado` o `Reprobado`.
4. Modifique el contador para que cambie de dos en dos.
5. Agregue al contador un botón **Cerrar**.

In [ ]:
# Ejercicio 1: escriba aquí su solución
set notas {4.5 3.8 2.9 4.0 5.0}

# Su código...

In [ ]:
# Ejercicios 2 y 3: escriba aquí su solución
# set curso ...
# proc clasificarNota {nota} { ... }

<details>
<summary><strong>Mostrar una solución posible de los ejercicios 1–3</strong></summary>

```tcl
set notas {4.5 3.8 2.9 4.0 5.0}
set suma 0.0
foreach nota $notas { set suma [expr {$suma + $nota}] }
set promedio [expr {$suma / [llength $notas]}]
set ordenadas [lsort -real $notas]
puts "Promedio: $promedio"
puts "Menor: [lindex $ordenadas 0]"
puts "Mayor: [lindex $ordenadas end]"

set curso [dict create nombre Compiladores grupo G3 estudiantes 19]

proc clasificarNota {nota} {
    if {$nota >= 4.5} { return Excelente }
    if {$nota >= 3.0} { return Aprobado }
    return Reprobado
}
```
</details>

# 12. Mapa mental final

Cuando lea una línea Tcl, pregunte siempre:

1. ¿Dónde termina el comando?
2. ¿Cuál es la primera palabra, es decir, el comando?
3. ¿Cuáles son sus argumentos?
4. ¿Qué sustituciones ocurren antes de invocarlo?
5. ¿Qué resultado devuelve?

Para Tk, añada tres preguntas:

1. ¿Qué widgets se crean?
2. ¿Cómo se distribuyen con `pack` o `grid`?
3. ¿Qué procedimiento se ejecuta ante cada evento?

## Lista de dominio 20/80

Marque cada punto cuando pueda explicarlo y modificarlo sin ayuda:

- [ ] `puts`, `set`, `$variable` y `[comando]`
- [ ] diferencia entre comillas y llaves
- [ ] `expr` con la expresión entre llaves
- [ ] `if`, `foreach`, `while` e `incr`
- [ ] listas y expansión `{*}`
- [ ] diccionarios
- [ ] procedimientos, argumentos y retorno
- [ ] `catch` y `error`
- [ ] widgets, `pack`, `grid`, `-command` y `bind`

Si domina esta lista, ya posee la base para estudiar archivos, namespaces, programación orientada a objetos, sockets, bases de datos, Picol y JimTcl.